# Конспект: pandas для олимпиады

**Маршрут:** загрузить → проверить → выбрать строки → обработать пропуски → посчитать → сохранить. `DataFrame` — таблица, `Series` — один столбец. Ниже `df` — таблица учеников из урока; замените файл и названия колонок на свои.

In [ ]:
import pandas as pd

df = pd.read_csv('../class04: numpy and pandas/students.csv')
print(df.shape)           # (строки, столбцы)
print(df.columns.tolist())
df.head()                 # первые 5 строк

## 1. Первые 30 секунд: что в таблице?

| Вопрос | Команда |
|---|---|
| Размер, первые и последние строки | `df.shape`, `df.head()`, `df.tail()` |
| Типы и заполненность | `df.info()`, `df.dtypes` |
| Числовые диапазоны | `df.describe()` |
| Пропуски по столбцам | `df.isna().sum().sort_values(ascending=False)` |
| Частоты категории | `df['city'].value_counts(dropna=False)` |
| Дубликаты строк | `df.duplicated().sum()` |

CSV с другим разделителем: `pd.read_csv('file.csv', sep=';')`. Текстовые числа превращайте в числа через `pd.to_numeric(..., errors='coerce')`; даты — через `pd.to_datetime(..., errors='coerce')`.

## 2. Выбор: столбцы, строки, условия

| Нужно | Код |
|---|---|
| Один столбец (`Series`) | `df['math_score']` |
| Несколько столбцов (`DataFrame`) | `df[['name', 'math_score']]` |
| По **позиции** (конец среза исключён) | `df.iloc[:5, :3]` |
| По **метке/имени** (конец среза включён) | `df.loc[0:4, ['name', 'city']]` |
| Условие + нужные столбцы | `df.loc[mask, ['name', 'math_score']]` |

Схема `loc`: **[какие строки, какие столбцы]**. У каждого условия с `&` / `|` нужны скобки; отрицание — `~`. Для проверки пропуска используйте `.isna()`, а не `== None`.

In [ ]:
mask = (df['grade'] == 11) & (df['math_score'] >= 80)
selected = df.loc[mask, ['name', 'city', 'math_score']].copy()
print(len(selected))

# Частые условия:
other = df.loc[df['city'].isin(['Москва', 'Казань']) & df['math_score'].between(70, 90)]

## 3. Столбцы и пропуски

`NaN` означает «неизвестно», а не ноль. Среднее pandas пропускает `NaN`. Для нескольких столбцов `.mean(axis=1)` считает **по строкам**; `.mean(axis=0)` — **по столбцам**. После фильтрации берите `.copy()`, затем меняйте копию.

In [ ]:
work = df.copy()
work['math_score'] = pd.to_numeric(work['math_score'], errors='coerce')
work['math_score'] = work['math_score'].fillna(work['math_score'].median())
work['average_score'] = work[['math_score', 'python_score']].mean(axis=1)
work['passed'] = work['average_score'] >= 70

print(work.isna().sum())
print(work['passed'].mean())  # доля True, от 0 до 1

**Для ML:** медиану, среднее, словарь кодирования вычисляйте только на обучающей части. Обработку train и validation/test удобно помещать в `Pipeline`, чтобы не было утечки.

## 4. Сортировка и агрегаты

- `df['math_score'].mean()`, `.median()`, `.sum()`, `.min()`, `.max()`, `.nunique()` — одно число.
- `df['city'].value_counts()` — сколько строк в каждой категории; `normalize=True` даёт доли.
- `df.sort_values('math_score', ascending=False).head(10)` — топ-10.
- `df.nlargest(10, 'math_score')` — короткий вариант топ-10.
- `df['math_score'].idxmax()` возвращает **метку строки**, её можно передать в `df.loc[...]`.

## 5. Группировка: типовой олимпиадный ответ

**Строки → группы → расчёт по каждой группе → сортировка.** `size` считает все строки группы, `count` — только непустые значения выбранного столбца. Среднее бинарного 0/1 — доля единиц.

In [ ]:
report = (work.groupby('city', dropna=False)
          .agg(n=('student_id', 'size'),
               mean_math=('math_score', 'mean'),
               passed_share=('passed', 'mean'))
          .sort_values('mean_math', ascending=False))
report.head()

## 6. Сводные таблицы и объединение

`pivot_table` — группы сразу по строкам и столбцам. `merge` — добавить признаки из второй таблицы по ключу. Если после `merge` строк неожиданно больше, ключ справа не уникален; `validate='m:1'` поймает такую ошибку.

In [ ]:
pivot = work.pivot_table(index='grade', columns='city', values='math_score', aggfunc='mean')
counts = pd.crosstab(work['grade'], work['city'])

city_map = pd.DataFrame({'city': ['Москва', 'Казань'], 'region': ['Центр', 'Поволжье']})
joined = work.merge(city_map, on='city', how='left', validate='m:1')
print(len(work), len(joined))  # число строк должно совпасть

### Ещё три полезных операции

```python
df['name'].str.contains('Анна', case=False, na=False)  # поиск по тексту
df['student_id'].map({1: 'A', 2: 'B'})               # значения по словарю
pd.concat([part1, part2], ignore_index=True)          # склеить таблицы по строкам
```

`df.drop_duplicates(subset=['student_id'])` убирает повторы по ключу; `reset_index(drop=True)` даёт индексы 0, 1, 2… после сортировки или фильтра.

## 7. Сохранить ответ без лишнего индекса

```python
answer = df[['student_id']].copy()     # сохраняем ID и порядок test
answer['prediction'] = predictions      # длина должна совпасть
assert len(answer) == len(df) and answer['prediction'].notna().all()
answer.to_csv('answer.csv', index=False)
```

**Частые ловушки:** `df['x']` и `df[['x']]` имеют разную форму; `&`/`|` требуют скобок; `count` не считает `NaN`; `merge` может размножить строки; `sort_values` возвращает новую таблицу; `to_csv` без `index=False` добавит лишний столбец.